 Preprocessing danych events i frames do formatu DataFrame wymaganego przez funkcję budującą pojedynczy graf dla zdarzenia `build_graph_for_event(event_row: pd.DataFrame, frame_row: pd.DataFrame, radius = 15.0)`
event_row:
id
match_id
index
type
player
team
possession_team
location
under_pressure

frame_row:
id	player_id	location	teammate	actor	keeper

In [18]:
from statsbombpy import sb
import numpy as np
import pandas as pd
from tqdm import tqdm

Wyciągnięcie meczów Mistrzostw Świata 2022

In [19]:
competitions = sb.competitions()

world_cup_competitions = competitions[competitions["competition_name"] == "FIFA World Cup"]
world_cup_22_id = world_cup_competitions.loc[competitions["season_name"] == "2022", "competition_id"].iloc[0]
world_cup_22_season_id = world_cup_competitions.loc[competitions["season_name"] == "2022", "season_id"].iloc[0]


matches = sb.matches(competition_id=world_cup_22_id,season_id=world_cup_22_season_id)

Definicja funkcji określającej kiedy nastąpiła strata piłki i dodanie jej do dataFrame'u

In [20]:
def add_turnover_label(events_model: pd.DataFrame, events_full: pd.DataFrame, min_events):
    events_full = events_full.copy()
    events_model = events_model.copy()

    events_full = events_full.sort_values("index").reset_index(drop=True)    
    id_to_pos = {
        event_id: pos
        for pos, event_id in enumerate(events_full["id"].tolist())
    }
    y_trunovers = []
    for _, row in events_model.iterrows():
        event_id = row["id"]

        if event_id not in id_to_pos:
            y_trunovers.append(0)
            continue
        pos = id_to_pos[event_id]
        current_possession_team = row["possession_team"]

        future = events_full.iloc[pos+1 : pos + 1 + min_events]

        turnover = 0 

        for _, future_event in future.iterrows():
            future_possession_team = future_event.get("possession_team")

            if pd.isna(future_possession_team):
                continue

            if future_possession_team != current_possession_team:
                turnover = 1
                break

        y_trunovers.append(turnover)

    events_model["y_turnover"] = y_trunovers

    return events_model
            

In [21]:
def add_direct_turnover_label(events_model: pd.DataFrame) -> pd.DataFrame:
    events_model = events_model.copy()

    y = []

    for _, row in events_model.iterrows():
        event_type = row["type"]

        turnover = 0

        if event_type == "Pass":
            # W StatsBomb udane podanie często ma pass_outcome = NaN
            if pd.notna(row.get("pass_outcome")):
                turnover = 1

        elif event_type == "Dribble":
            if row.get("dribble_outcome") == "Incomplete":
                turnover = 1

        elif event_type == "Ball Receipt*":
            if pd.notna(row.get("ball_receipt_outcome")):
                turnover = 1

        elif event_type == "Carry":
            # carry samo w sobie zwykle nie ma outcome
            turnover = row.get("y_turnover", 0)

        y.append(turnover)

    events_model["y_direct_turnover"] = y

    return events_model

Funkcja budująca model dataFrame z wymaganymi danymi

In [22]:
def df_model(matches):
    BALL_EVENTS = [
        "Pass",
        "Carry",
        "Ball Receipt*",
        "Dribble",
    ]

    needed_cols = [
        "id",
        "match_id",
        "index",
        "period",
        "timestamp",
        "minute",
        "second",
        "type",
        "team",
        "player",
        "possession",
        "possession_team",
        "location",
        "under_pressure",
    ]

    events_list = []
    frames_list = []

    for match_id in tqdm(matches["match_id"],total= len(matches)):
        events = sb.events(match_id=match_id)

        events_model = events.loc[
            events["type"].isin(BALL_EVENTS),
            needed_cols
        ].copy()

        events_model = add_turnover_label(
            events_model=events_model,
            events_full=events,
            min_events=3,
        )

        events_model = add_direct_turnover_label(events_model=events_model)

        events_model["under_pressure"] = (
            events_model["under_pressure"]
            .replace({np.nan: False})
            .astype(bool)
        )

        events_model["y_pressure"] = events_model["under_pressure"].astype(int)

        frames = sb.frames(match_id=match_id)

        frames_model = frames.loc[
            frames["id"].isin(events_model["id"])
        ].copy()

        events_list.append(events_model)
        frames_list.append(frames_model)

    events_model_df = pd.concat(events_list, ignore_index=True)
    frames_model_df = pd.concat(frames_list, ignore_index=True)

    return events_model_df, frames_model_df

Budowa modelu

In [23]:
import warnings

warnings.filterwarnings("ignore")

In [24]:
events_model, frames_model = df_model(matches)

100%|██████████| 64/64 [03:18<00:00,  3.10s/it]


Zapis dataFreme'ów do .csv

In [25]:
events_model.to_csv('events_model_df.csv', index=False, encoding='utf-8')
frames_model.to_csv('frames_model_df.csv', index=False, encoding='utf-8')